# MotherDuck Raw Data EDA

This notebook connects to MotherDuck to explore the raw (Bronze) data collected from OLX.

In [ ]:
import os
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Load environment variables
load_dotenv("../.env")

token = os.getenv("MOTHERDUCK_TOKEN")
database = os.getenv("MD_DATABASE", "my_db")

if not token:
    print("Error: MOTHERDUCK_TOKEN not found in environment variables.")

## 1. Connect to MotherDuck

In [ ]:
con = duckdb.connect(f"md:{database}?token={token}")
print(f"Connected to MotherDuck database: {database}")

## 2. Basic Stats

In [ ]:
# List tables
con.query("SHOW ALL TABLES").show()

In [ ]:
# Row counts per layer/mode
con.query("""
    SELECT 'rent' as mode, count(*) as count FROM bronze.rent_bronze
    UNION ALL
    SELECT 'sale' as mode, count(*) as count FROM bronze.sale_bronze
""").to_df()

## 3. Explore Rent Data

In [ ]:
# Sample rent data
rent_df = con.query("SELECT * FROM bronze.rent_bronze LIMIT 1000").to_df()
rent_df.head()

In [ ]:
# Inspect snapshot dates
rent_df['snapshot_date'].value_counts().sort_index().plot(kind='bar', title='Listings by Snapshot Date')
plt.show()

## 4. Deep Dive into Data

The tables already have parsed columns like `price_total`, `city`, and `title`. Let's use them.

In [ ]:
# Extract some basic fields
enriched_df = con.query("""
    SELECT 
        source_listing_id,
        snapshot_date,
        title,
        price_total,
        city,
        district
    FROM bronze.rent_bronze
    LIMIT 5000
""").to_df()

enriched_df.head()

In [ ]:
# Price distribution by city
plt.figure(figsize=(12, 6))
sns.boxplot(data=enriched_df, x='city', y='price_total')
plt.title('Rent Price Distribution by City (Sample)')
plt.xticks(rotation=45)
plt.show()

## 5. Cleanup

In [ ]:
con.close()